In [1]:
!pip install mlflow --quiet

import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
import pandas as pd

# Cargar dataset
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

In [2]:
mlflow.set_experiment("rf_experimento_california")

2026/04/10 16:38:18 INFO mlflow.tracking.fluent: Experiment with name 'rf_experimento_california' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/idavid/dev/tecmilenio/aplicaciones-ciencia-datos/temas/Practica-tema-6/mlruns/2', creation_time=1775864298780, experiment_id='2', last_update_time=1775864298780, lifecycle_stage='active', name='rf_experimento_california', tags={}, trace_location=None, workspace='default'>

In [3]:
params_grid = [
{"n_estimators": 50, "max_depth": 5},
{"n_estimators": 100, "max_depth": 5},
{"n_estimators": 100, "max_depth": 10},
{"n_estimators": 200, "max_depth": 10},
{"n_estimators": 200, "max_depth": None},
{"n_estimators": 300, "max_depth": None},
]

In [4]:
for params in params_grid:
  with mlflow.start_run(run_name=f"rf_ne{params['n_estimators']}_md{params['max_depth']}"):

    # Registrar parámetros
    mlflow.log_param("n_estimators", params["n_estimators"])
    mlflow.log_param("max_depth", params["max_depth"])

    # Entrenar modelo
    model = RandomForestRegressor(
    n_estimators=params["n_estimators"],
    max_depth=params["max_depth"],
    random_state=42
    )
    model.fit(X_train, y_train)

    # Registrar métrica
    r2 = model.score(X_test, y_test)
    mlflow.log_metric("r2", r2)

    # Guardar modelo entrenado
    mlflow.sklearn.log_model(model, "model_rf")

  print(f"Run: {params} → R² = {r2:.4f}")

2026/04/10 16:38:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 16:38:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run: {'n_estimators': 50, 'max_depth': 5} → R² = 0.6469


2026/04/10 16:38:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 16:38:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run: {'n_estimators': 100, 'max_depth': 5} → R² = 0.6468


2026/04/10 16:38:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 16:38:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

Run: {'n_estimators': 200, 'max_depth': 10} → R² = 0.7748


2026/04/10 16:39:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 16:39:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run: {'n_estimators': 200, 'max_depth': None} → R² = 0.8062


2026/04/10 16:40:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 16:40:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run: {'n_estimators': 300, 'max_depth': None} → R² = 0.8066


In [5]:
exp = mlflow.get_experiment_by_name("rf_experimento_california")
experiment_id = exp.experiment_id
df_runs = mlflow.search_runs(experiment_ids=[experiment_id])
df_summary = df_runs[
["run_id", "params.n_estimators", "params.max_depth", "metrics.r2"]
].sort_values("metrics.r2", ascending=False)

df_summary

,run_id,params.n_estimators,params.max_depth,metrics.r2
0,44ccae53417a457fb4acbdd44b71431c,300,None,0.806600
1,64ac273e696f4873aa158bedf12f1344,200,None,0.806186
2,30c6c52405e2444cb15f49b83203e31c,200,10,0.774774
3,c56fe783422241849199c552fdb20048,100,10,0.773740
5,be9e3287982e47b2b011318c5ee6e3aa,50,5,0.646870
4,faade4ba04ec402f98fd247ba279f684,100,5,0.646831


In [6]:
best_run = df_summary.iloc[0]
best_run_id = best_run.run_id
best_run

run_id                 44ccae53417a457fb4acbdd44b71431c
params.n_estimators                                 300
params.max_depth                                   None
metrics.r2                                       0.8066
Name: 0, dtype: object

In [7]:
model_uri = f"runs:/{best_run_id}/model_rf"
best_model = mlflow.sklearn.load_model(model_uri)
# Realizar predicciones de prueba
best_model.predict(X_test[:5])

array([0.49249333, 0.73544   , 4.87372677, 2.52624   , 2.30335   ])